# Who should get the offer: learned rules and random offers with TMLE

## At a glance

A hospital offers navigation to some of the patients it sends home, and nobody randomized the
offer. Navigation helps patients with high need and harms patients with low need. Who should get
it, and if staff can cover only some of those who qualify, what does a random offer among them
achieve?

The rule learned from these records offers navigation mainly above a need score of 0, and under it
71.7% of patients would stay home for 30 days (95% confidence interval 69.0% to 74.3%). Under
current practice 67.0% did, a gap that `cleverly` does not yet give an interval. Offering
navigation at random to half of the patients with a need score above 0 raises the share by 2.6
per 100 patients, against offering it to no one (95% CI 1.1 to 4.2).

Both answers are causal only if the recorded need score and living situation capture every common
cause of the offer and of staying home, which the data cannot check. In these simulated data each
interval contains its true value (71.7 and 3.3 per 100).

## The question

At the discharge-home order, the program decides whether to offer a patient navigation: a
bedside transition plan and two navigator contacts in the 30 days after discharge. The outcome is
whether the patient stays home, with no unplanned readmission, for 30 days. A treatment rule
decides from a patient's recorded characteristics who gets the offer, and its value is the share
who would stay home had every offer followed that rule.

```text
  need, lives_alone ──▶ offer ──▶ stays home for 30 days
  need, lives_alone ─────────────▶ stays home for 30 days
  need also changes how much the offer helps
  assumed absent: an unrecorded cause of both the offer and staying home
```

Staff offered navigation more often to patients with higher need and to those who live alone, and
both also change the chance of staying home, so they are confounders: variables that drive both
the offer and the outcome. Because need also changes how much the offer helps, the best rule
offers navigation to some patients only.

## The data

Every fit sets its seeds, so a rerun prints the same numbers.

In [1]:
from dataclasses import replace

import numpy as np
import pandas as pd

import cleverly

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
print("cleverly", cleverly.__version__)

cleverly 0.2.8


Another version of `cleverly` can print slightly different numbers.

`make_treatment_rules` draws 2,000 simulated patients and also returns the true answers, which
real data never give you.

In [2]:
from cleverly.datasets import make_treatment_rules

# With your own data, load one row per patient, such as pd.read_csv(...), with the recorded
# characteristics, the offer (0 or 1), and the outcome (1 if the patient stayed home 30 days).
# Real data have no `truth`: delete the lines marked "simulation only" here and below. The cells
# below name need, lives_alone, offer and home_30, and cut need at 0 and at the bands of the rule
# cell; change those for your data.
frame, truth = make_treatment_rules(n=2_000, seed=7)
print("rows and columns:", frame.shape)
print(list(frame.columns))
print(frame.head().round(2).to_string())
print("offered:", int(frame["offer"].sum()), " stayed home 30 days:", int(frame["home_30"].sum()))
print()

# simulation only, to the end of the cell
print("true share who stay home for 30 days:")
print(f"  offer to no one     {truth['ey_none']:.4f}")
print(f"  offer to everyone   {truth['ey_all']:.4f}")
best, cutoff = truth["ey_best_rule"], truth["best_rule_cutoff"]
print(f"  the best rule       {best:.4f}, offers when need > {cutoff:g}")

rows and columns: (2000, 4)
['need', 'lives_alone', 'offer', 'home_30']
   need  lives_alone  offer  home_30
0  0.25          0.0    0.0      1.0
1  0.79          0.0    1.0      1.0
2  0.55          1.0    1.0      1.0
3 -0.55          1.0    1.0      1.0
4 -0.40          1.0    1.0      1.0
offered: 1061  stayed home 30 days: 1339

true share who stay home for 30 days:
  offer to no one     0.6522
  offer to everyone   0.6763
  the best rule       0.7197, offers when need > -0.1


Each row is one patient. `need` is a need score from -1 (lowest) to 1 (highest), where 0 is the
middle of the scale rather than a clinical cut-off, and `lives_alone` is 1 for a patient who lives
alone. Offering navigation to everyone would raise the share who stay home only from 0.6522 to
0.6763, while the best rule, which offers it only where it helps, would reach 0.7197.

## Why a simple comparison misleads

In [3]:
print("average need and share living alone, by the offer:")
print(frame.groupby("offer")[["need", "lives_alone"]].mean().round(3))
print()

# The simple comparison: the share who stayed home, offered minus not offered.
shares = frame.groupby("offer")["home_30"].mean()
print(f"simple difference, everyone: {shares[1] - shares[0]:+.4f}")
print(f"true difference, everyone offered against none: {truth['ate']:+.4f}")  # simulation only
half = np.where(frame["need"] > 0, "need above 0", "need 0 or below")
by_half = frame.groupby([half, "offer"])["home_30"].mean().unstack()
print("simple difference by half:", (by_half[1.0] - by_half[0.0]).round(4).to_dict())

average need and share living alone, by the offer:
        need  lives_alone
offer                    
0.0   -0.096        0.473
1.0    0.084        0.524

simple difference, everyone: -0.0328
true difference, everyone offered against none: +0.0241
simple difference by half: {'need 0 or below': -0.1467, 'need above 0': 0.1066}


Offered patients had higher need and lived alone more often, and without navigation both lower the
chance of staying home. So the simple difference, -0.0328, says navigation hurts, while offering it
to everyone would in truth raise the share by 0.0241. Split at a need score of 0, the differences
point opposite ways, -0.1467 and 0.1066: a first sign that the offer helps only some patients,
though each half still mixes the offer's effect with who got it.

## Set up the analysis

First, describe the study (`StudyProtocol`): who is in it, when follow-up starts, what each option
means, and what the outcome is. Each result keeps a copy as a record of what the estimate means.
The code adapts the program's [shared study description](index.md#the-shared-study-design).

In [4]:
from cleverly.datasets import navigation_protocol

# For your own study, build cleverly.StudyProtocol(target_population=..., eligibility=...,
# time_zero=..., treatment_strategies=(...), treatment_versions=(...), outcome=..., horizon=...,
# intercurrent_event_handling=(...), interference_unit=..., assumption_rationale=(...)),
# one field for each printed label, with your own text.
program = navigation_protocol()
protocol = replace(
    program,
    treatment_strategies=(
        *program.treatment_strategies,
        "Offer navigation by a rule learned from the recorded need and living situation",
        "Offer navigation at random, with chance one half, to patients with a need score above 0",
    ),
    treatment_versions=(
        *program.treatment_versions,
        "The standard offer for the patients the rule selects, usual support for the others",
        "The standard offer for the patients the random draw selects, usual support for the others",
    ),
    outcome="Stayed home for 30 days after discharge, with no unplanned readmission",
    intercurrent_event_handling=(
        "Count an unplanned readmission within 30 days as not staying home",
        program.intercurrent_event_handling[1],  # the shared rule on completed contacts
        "Count death before day 30 as not staying home (composite strategy)",
    ),
)
print("\n".join(protocol.summary_lines()))

Population: Adults with a discharge-home order at a participating hospital during the enrollment period
Eligibility: Age 18 years or older; Discharge home ordered at a participating hospital
Follow-up starts: Discharge-home order, after baseline measurement and before the navigation offer
Strategies: Offer standard transition navigation; Provide usual discharge support; Offer navigation by a rule learned from the recorded need and living situation; Offer navigation at random, with chance one half, to patients with a need score above 0
Versions: Bedside transition plan and two scheduled navigator contacts within 30 days; No access to the transition-navigation offer; The standard offer for the patients the rule selects, usual support for the others; The standard offer for the patients the random draw selects, usual support for the others
Outcome: Stayed home for 30 days after discharge, with no unplanned readmission
Horizon: 30 days after discharge
Intercurrent events: Count an unplanned

The protocol adds the learned rule and the random offer to the shared description's two options.
It counts a readmission or a death after discharge as not staying home, so an option cannot look
better because its patients die first.

## Name the columns, the question and its conditions

`PointTreatment` names the offer, the outcome and the characteristics to adjust for.
`LearnedRuleValue` asks for the value of a rule that the fit learns from the data, and `identify`
lists the conditions the answer needs.

In [5]:
from cleverly import CausalStudy, LearnedRuleValue, PointTreatment

# For your own data, change the column names.
design = PointTreatment(outcome="home_30", treatment="offer", adjustment=("need", "lives_alone"))
study = CausalStudy(frame, design=design, protocol=protocol)
question = study.identify(LearnedRuleValue())
print(question.summary())

                      Causal question (ey_learned_rule)
Question:      weighted average over    Rows:                            2,000
               the outer folds of the   Answered by:  adjusting for covariates
               value of the rule
               learned on each fold's
               training rows
Outcome:       home_30 (binary)
Treatment:     offer (levels 0, 1)
Adjusted for:  need, lives_alone
Conditions the answer rests on
condition                checkable   what it says
------------------------------------------------------------------------------
well-defined treatment   no          Each unit's observed outcome is the
                                     outcome it would have had under the
                                     treatment it actually received. This
                                     needs the treatment to be well defined:
                                     every way of getting the same level has
                                     the same effect

The question line is the target: the average value of the rules learned with each tenth of the
patients held out in turn (the "outer folds"). The data cannot check any of the three conditions:

- well-defined treatment (consistency): an offer means the same contacts for every patient;
- no interference: one patient's offer does not change another's chance of staying home;
- no hidden confounding: need and living alone hold every common cause of the offer and of
  staying home.

:::{dropdown} Details: the further conditions
Note [1] points to seven more conditions in `summary(detail="full")`. Two matter most here:
overlap, meaning every patient had a real chance of either option, which the weight check below
partly checks; and a clear best rule, meaning the offer has some effect, good or bad, for almost
every patient, so the learned rules do not flip at random. The simulation meets both. On real
data nothing checks the clear best rule directly. Only TMLE, used next, answers this question.
:::

## Estimate the value of a learned rule

The fit splits the patients into ten folds of 200. For each fold, it fits an outcome model on the
other nine, and the rule offers navigation where that model predicts a better chance of staying
home with the offer. It scores that rule on the held-out fold with targeted maximum likelihood
estimation (TMLE), a correction step that uses the treatment model to remove the bias the outcome
model leaves, and averages the ten fold scores.

- `outcome_learner` predicts staying home. Its products of two inputs, such as offer times need,
  let the offer's effect change with need.
- `treatment_learner` predicts who was offered.
- `CrossFitting(n_folds=10, fold_evaluation=True)` scores each fold's rule only on that fold.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

from cleverly import CrossFitting, ModelSpec, Runtime, TMLEMethod

# For your own data, give the outcome model the offer-by-characteristic terms that the rule can
# use. C=1e6 turns off scikit-learn's default shrinkage of the coefficients, which would pull the
# offer's effect toward zero and blur where it helps. Runtime fixes the seeds.
outcome_model = make_pipeline(
    PolynomialFeatures(2, interaction_only=True, include_bias=False),
    LogisticRegression(C=1e6, max_iter=5000),
)
method = TMLEMethod(
    models=ModelSpec(
        outcome_learner=outcome_model,
        treatment_learner=LogisticRegression(C=1e6, max_iter=1000),
    ),
    cross_fitting=CrossFitting(n_folds=10, fold_evaluation=True),
    runtime=Runtime(random_state=13, n_jobs=1),
)
learned = question.estimate(method=method)
print(learned.summary())

                            Effect estimate (TMLE)
Question:      value of a treatment     Rows:                            2,000
               rule learned from the    Treated share:                   0.530
               data                     Out-of-fold predictions:      10 folds
Outcome:       home_30 (binary)         Outcome model:     logistic regression
Adjusted for:  need, lives_alone                        after polynomial terms
Method:        targeted maximum         Treatment model:   logistic regression
               likelihood (TMLE)        Probability limits:  [0.01471, 0.9853]
                               estimate  std. error          95% CI  p-value
------------------------------------------------------------------------------
ey_learned_rule[learned rule]     0.717       0.014  [0.690, 0.743]  <0.0001
Notes:
[1] The interval of ey_learned_rule[learned rule] is built on the log-odds
    scale and turned back, so it stays inside the possible range of a
    proport

The estimate row is the value of the learned rule: 0.717, so about 72 of every 100 patients would
stay home for 30 days (95% CI 0.690 to 0.743).

:::{dropdown} Details: the other numbers in the summary
The p-value tests only whether the share is zero, and the treated share, 0.530, is the share
offered in the data. The probability limits are a floor and a ceiling for each predicted chance of
the offer, chosen from the number of rows, and no patient reached them. Note [1] says each interval
is built on the log-odds scale (the scale of a logistic regression) and turned back, so it stays
between 0 and 1 and can sit slightly off-centre around the estimate
([Reading a summary](../user-guide/results-assessment.md#reading-a-summary)).
:::

In [7]:
from cleverly.datasets import learned_rule_truth

value = learned["ey_learned_rule[learned rule]"]
record = learned.extra["learned_rule"]
low, high = value.ci
print(f"value of the learned rule: {value.psi:.4f}, 95% CI {low:.4f} to {high:.4f}")
print(
    f"per 100 patients: {100 * value.psi:.1f} stay home, 95% CI {100 * low:.1f} to {100 * high:.1f}"
)
print(f"stayed home under the staff's own choices: {100 * frame['home_30'].mean():.1f} per 100")
(split,) = record.splits  # the fit drew one split of the patients into ten folds
# arm_shares holds, per fold, the share of patients its rule does not offer and offers.
fold_shares = [offered for _, offered in split.arm_shares]
print(
    f"share of patients each fold's rule offers: {min(fold_shares):.3f} to {max(fold_shares):.3f}"
)
# learned.nuisance holds the fitted parts behind the estimate. Its regimes.values[i, a, r] is
# patient i's chance of offer a under option r. The learned rule is
# option 0, so [:, 1, 0] is 1 when the rule from the fold that held the patient out offers.
offered_by_rule = pd.Series(learned.nuisance.regimes.values[:, 1, 0], index=frame.index)
edges = [-1.0, -0.1, 0.0, 0.1, 0.2, 1.0]
labels = ["-1 to -0.1", "-0.1 to 0", "0 to 0.1", "0.1 to 0.2", "0.2 to 1"]
bands = pd.cut(frame["need"], edges, labels=labels, include_lowest=True).rename("need score")
print("share the rules offer, by need score:")
print(offered_by_rule.groupby(bands, observed=True).mean().round(3).to_string())
print()

# simulation only, to the end of the cell: the true value of the ten fold rules, which the
# simulation computes by refitting each fold's outcome model.
true_target = learned_rule_truth(learned, outcome_model)
print(f"true value of the ten fold rules: {true_target:.4f}, or {100 * true_target:.1f} per 100")
print(f"true value of the best rule:      {truth['ey_best_rule']:.4f}")

value of the learned rule: 0.7174, 95% CI 0.6900 to 0.7433
per 100 patients: 71.7 stay home, 95% CI 69.0 to 74.3
stayed home under the staff's own choices: 67.0 per 100
share of patients each fold's rule offers: 0.345 to 0.545
share the rules offer, by need score:
need score
-1 to -0.1    0.000
-0.1 to 0     0.009
0 to 0.1      0.300
0.1 to 0.2    0.744
0.2 to 1      1.000



true value of the ten fold rules: 0.7174, or 71.7 per 100
true value of the best rule:      0.7197


The rules offer navigation to almost no patient with a need score below -0.1 and to almost every
patient above 0.2. The best rule offers it above -0.1, so the rules miss many patients between -0.1
and 0.2, where navigation helps only a little. Their true value, 0.7174, is just below the best
rule's 0.7197, and the estimate lands on it.

To deploy a rule, you would fit the outcome model once on all patients and offer
navigation where it predicts a gain. When the fold rules settle on one rule, as the conditions
below require, that rule settles on the same one, so the fold rules' value is the guide to it. Its
true value here is 0.7175 (the logistic row of the in-sample comparison below).

:::{dropdown} Details: when the interval can be trusted
The interval is valid in large samples when every patient had a real chance of either option, the
treatment model or the outcome model is good enough, and the fold rules settle on one rule as the
sample grows. The last condition fails when the offer has no effect for a share of patients: the
rules then flip at random, and the interval can be too narrow. The simulation studies also find
the interval too narrow at 2,000 patients when the effect is weak for everyone (see "Go deeper").

The share each fold's rule offers varies with which 200 patients the fold holds and where its rule
draws the line between -0.1 and 0.2.

`CrossFitting(enabled=False)` estimates the value of the one rule fitted on all patients, with an
interval, when the outcome model has a fixed set of terms, as this one does. The "Learned
treatment rules" section of the [estimands guide](../user-guide/estimands.md) shows how.
:::

## Check the weights behind the estimate

The correction step weights each patient whose offer matches the rule by one over their predicted
chance of that offer, so every patient needs a real chance of the option the rule assigns.

In [8]:
assessment = learned.assess()
ledger = assessment.to_frame()
# Keep the checks that ran: `passed` has a pass mark, `completed` is a report to read.
ran = ledger[ledger["status"].isin(["passed", "completed"])]
print(ran[["check", "status"]].to_string(index=False))
print("needs attention:", tuple(item.name for item in assessment.attention) or "none")
print()
overlap = assessment.report("support").to_frame().iloc[0]
print(f"smallest predicted chance of the offer the rule assigns: {overlap['min_propensity']:.3f}")
print(f"largest weight: {overlap['max_ratio']:.2f}")
matched = int((frame["offer"] == offered_by_rule).sum())
print(f"patients whose offer matched the rule: {matched} of {len(frame)}")
print(f"effective sample size: {overlap['effective_n']:.0f}")

              check    status
    score_equations    passed
            support completed
    nuisance_models completed
omitted_confounding completed
   robustness_value completed
           elements completed
            contour completed
needs attention: none

smallest predicted chance of the offer the rule assigns: 0.429
largest weight: 2.33
patients whose offer matched the rule: 1152 of 2000
effective sample size: 1139


Nothing needs attention, and `score_equations passed` says the correction step finished. Every
patient had a predicted chance of at least 0.429 of the option the rule assigns, so the largest
weight is 2.33. The weights of the 1152 patients whose offer matched the rule are worth 1139
equally weighted patients, the effective sample size. Warning signs are a chance near 0, a few
huge weights, or an effective sample size far below the number matched.

A clean check cannot show that the treatment model is right. The rows from `omitted_confounding`
to `contour` belong to the missing-confounder section below.

## What goes wrong when a rule is scored on the patients it was learned from

The common mistake is to fit one outcome model on all patients, read off the rule, and score it
with the same model on the same patients. The code does that by hand with the logistic model above
and with a random forest, a flexible model that can memorize noise.

In [9]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier

from cleverly.datasets import treatment_rule_value

# thread_limit runs the forest on one CPU thread, so reruns match.
from cleverly.learners import thread_limit

inputs = ["offer", "need", "lives_alone"]


def with_offer(data, offer):
    return data[["need", "lives_alone"]].assign(offer=float(offer))[inputs]


forest = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=17, n_jobs=1)
rows = []
with thread_limit():
    for label, model in (("logistic, as above", outcome_model), ("random forest", forest)):
        fitted = clone(model).fit(frame[inputs], frame["home_30"])

        def chance(data, offer, fitted=fitted):
            return fitted.predict_proba(with_offer(data, offer))[:, 1]

        def rule(data, chance=chance):
            return (chance(data, 1) > chance(data, 0)).astype(float)

        offered = rule(frame)
        score = np.where(offered == 1, chance(frame, 1), chance(frame, 0)).mean()
        rows.append(
            {
                "model": label,
                "share offered": offered.mean(),
                "in-sample score": score,
                "true value": treatment_rule_value(rule),  # simulation only
            }
        )
scored = pd.DataFrame(rows)
print(scored.round(4).to_string(index=False))

             model  share offered  in-sample score  true value
logistic, as above         0.4475           0.7303      0.7175
     random forest         0.4590           0.7746      0.6979


Both rules score better on the patients they were learned from than their true values, the forest
by far: 0.7746 against 0.6979. It offers navigation to patients who happened to do well with it,
withholds it from those who happened to do well without it, and counts that luck as success, so
its score even outranks the better rule's.

Across 400 simulated samples (see "Go deeper") the forest's in-sample score was too high by 0.0871
on average, and the logistic one by 0.0036. For a flexible model, no standard interval formula
allows for the same patients both choosing and scoring the rule, which is why the fit above scores
each rule only on patients it never saw.

## When capacity is limited: offer at random

Suppose staff can cover only about a quarter of the patients. The program would then offer
navigation with chance one half to each patient with a need score above 0, roughly half of them,
and to no one else, fixed before anyone looks at an outcome.

`Stochastic(..., density_kind="known")` declares that the program set these chances, so the
interval treats them as exact. `Static(0)` offers to no one, and `reference=` makes it the baseline,
which means the same anywhere, unlike current practice. `RegimeMean` asks for the share who stay home
under each option. `CrossFitting(enabled=False)` fits every model on every patient, which suits
simple models that cannot memorize their rows.

In [10]:
from cleverly import RegimeMean
from cleverly.interventions import Static, Stochastic


# For your own program, return the chances your random offer uses: one row per patient, with the
# chance of no offer and then the chance of an offer.
def random_chances(data):
    offer = np.where(data["need"] > 0, 0.5, 0.0)
    return np.column_stack([1 - offer, offer])


plans = (
    Static(0, name="offer to none"),
    Stochastic(random_chances, name="random offer", density_kind="known"),
)
in_sample = replace(method, cross_fitting=CrossFitting(enabled=False))
options = study.identify(RegimeMean(plans, reference="offer to none")).estimate(method=in_sample)
print(options.summary())

                            Effect estimate (TMLE)
Question:      effect of offer on       Rows:                            2,000
               home_30; each row is     Treated share:                   0.530
               defined below            Out-of-fold predictions:            no
Outcome:       home_30 (binary)         Outcome model:     logistic regression
Adjusted for:  need, lives_alone                        after polynomial terms
Method:        targeted maximum         Treatment model:   logistic regression
               likelihood (TMLE)        Probability limits:  [0.01471, 0.9853]
                           estimate   std. error           95% CI    p-value
------------------------------------------------------------------------------
ey_regime[offer to none]      0.668        0.016   [0.636, 0.698]    <0.0001
ey_regime[random offer]       0.694        0.012   [0.669, 0.717]    <0.0001
What each row estimates
                           meaning
---------------------------

Each row is the share who would stay home under one option (the counterfactual mean under a
regime): 0.668 with no offers and 0.694 under the random offer. Read the intervals rather than the
simultaneous bands, which `cleverly` marks experimental. Warning [1] concerns flexible models,
not this fit.

In [11]:
names = ["ey_regime[random offer]", "ey_regime[offer to none]"]
# psi holds the named estimates in order; .psi is an estimate's value and .ci its interval.
gain = options.contrast(lambda psi: psi[0] - psi[1], names, name="random offer minus none")
low, high = gain.ci
print(f"random offer minus no offers: {gain.psi:+.4f}, 95% CI {low:+.4f} to {high:+.4f}")
print(f"per 100 patients: {100 * gain.psi:+.1f}, 95% CI {100 * low:+.1f} to {100 * high:+.1f}")
options_assessment = options.assess()
print("needs attention:", tuple(item.name for item in options_assessment.attention) or "none")
random_overlap = options_assessment.report("support").to_frame().set_index("regime")
print("largest weight, by option:", random_overlap["max_ratio"].round(2).to_dict())
print()

# simulation only, to the end of the cell: the true gain
true_random = treatment_rule_value(lambda data: random_chances(data)[:, 1])
true_gain = true_random - truth["ey_none"]
print(f"true gain: {true_gain:+.4f}, or {100 * true_gain:+.1f} per 100")

random offer minus no offers: +0.0262, 95% CI +0.0113 to +0.0422
per 100 patients: +2.6, 95% CI +1.1 to +4.2
needs attention: none
largest weight, by option: {'offer to none': 3.19, 'random offer': 2.26}

true gain: +0.0335, or +3.3 per 100


The random offer raises the share who stay home by 2.6 per 100 patients against no offers (95% CI
1.1 to 4.2), and the true gain is 3.3. The standard error of `options.contrast(...)` allows for
both shares coming from the same patients. The random offer gains more than offering navigation to
everyone (0.0241 in truth, from the data step), because it reaches only patients with need above 0,
whom in this simulation the offer always helps. Its weight check is clean (largest weight
3.19).

## How much could a missing confounder move the answers?

The robustness value asks how strong a missing confounder would have to be to erase the random
offer's gain, so the cell refits with `RegimeContrast`, which makes the gain a row of the fit. A
benchmark uses a recorded variable as a yardstick for that strength.

In [12]:
from cleverly import RegimeContrast

random_fit = study.identify(RegimeContrast(plans, reference="offer to none")).estimate(
    method=in_sample
)
RANDOM_OFFER = "ate_regime[random offer vs offer to none]"
sensitivity = random_fit.sensitivity
robustness = sensitivity.robustness_value(estimand=RANDOM_OFFER)
print(f"robustness value: {robustness['rv']:.3f} ({robustness['rva']:.3f} at the 95% limit)")
# For your own data, benchmark each recorded confounder that the random offer does not read. A
# benchmark refits without the variable, and the random offer cannot assign offers without need.
benchmark = sensitivity.benchmark(covariates=("lives_alone",), estimand=RANDOM_OFFER)
print(
    f"strengths of lives_alone: outcome side {benchmark.cf_y:.3f}, offer side {benchmark.cf_d:.3f}"
)
bound = sensitivity.omitted_confounding(estimand=RANDOM_OFFER, benchmark=benchmark)
print(
    f"a confounder as strong as lives_alone: largest shift {bound.bias:.4f}, "
    f"95% limits {bound.ci_lower:.4f} to {bound.ci_upper:.4f}"
)

robustness value: 0.076 (0.040 at the 95% limit)
strengths of lives_alone: outcome side 0.003, offer side 0.002
a confounder as strong as lives_alone: largest shift 0.0008, 95% limits 0.0128 to 0.0404


A missing confounder has two strengths, each a share from 0 to 1: on the outcome side, the share of
the outcome's unexplained variation it explains; on the offer side, roughly how much better it
would predict who got an offer beyond need and living alone. The robustness value, 0.076, is a
floor: a confounder weaker than that on both sides cannot erase the gain, though one strong on a
single side can. At 0.040 on both sides, it pulls the lower end of the interval to zero.

Living alone has strengths of only 0.003 and 0.002, and a confounder that strong would shift the
gain by at most 0.0008 (one-sided 95% limits 0.0128 and 0.0404). Whether unrecorded support at home
could reach 0.040 on both sides is a judgment about the program.

:::{dropdown} Details: the limits of the bound
The bound reads the random offer's gain, under the worst case in which the confounder's two effects
align. The offer-side strength is measured through the weights that correct for who got an offer.
The one-sided limits sit inside the two-sided interval because the shift is tiny. No simulation
study covers the bound for a random offer, so read it as a description. The check cannot leave out
`need`, because the random offer reads it.
:::

In [13]:
# The same question for the learned rule's value, with the ten rules the fit learned held fixed.
RULE = "ey_learned_rule[learned rule]"
shift = 0.05  # For your own program, the drop in the value that would change the decision.
rule_sensitivity = learned.sensitivity
rule_robustness = rule_sensitivity.robustness_value(
    estimand=RULE, null_hypothesis=value.psi - shift
)
print(f"strength on both sides that could lower the value by {shift}: {rule_robustness['rv']:.3f}")
rule_bound = rule_sensitivity.omitted_confounding(estimand=RULE, cf_y=0.03, cf_d=0.03)
print(
    f"with both strengths at 0.03: largest shift {rule_bound.bias:.4f}, "
    f"value between {rule_bound.lower:.4f} and {rule_bound.upper:.4f}"
)

strength on both sides that could lower the value by 0.05: 0.079
with both strengths at 0.03: largest shift 0.0185, value between 0.6989 and 0.7359


For the learned rule's value, with the ten learned rules held fixed, the robustness value is
0.079: a confounder weaker than that on both sides cannot lower the value by 0.05 (5 per 100
patients), though one strong on a single side can. With both strengths at 0.03, the value could
move by at most 0.0185, to between 0.6989 and 0.7359, with no 95% limits. A benchmark is not
possible, because leaving out a variable would change the rules.

:::{dropdown} Details: what the rule's bound holds fixed
The reported value is the average of the ten fold rules' values, so the bound treats the ten rules
as fixed and asks how far a missing confounder could bias the estimate of that average, not how
the rules were learned. No formula for the uncertainty of these shifts is worked out for a learned
rule, and no simulation study covers this bound, so read it as a description. The
[technical reference](../technical-reference/validation-methods.md#omitted-variable-bounds-robustness-value-benchmark-and-contours)
gives the derivation and the conditions it needs, under "The point bound of a learned-rule value".
:::

## What we can and can't conclude

The learned rule's value (71.7% staying home for 30 days, 95% interval 69.0% to 74.3%, averaged
over the ten versions the fit learned) and the random offer's gain over no offers (2.6 per 100,
95% interval 1.1 to 4.2) hold for adults sent home from a participating hospital, if the
conditions under "Name the columns, the question and its conditions" hold and the models are good
enough.

Neither answer says what the best possible rule achieves. The rule's interval can be too narrow if
the offer has no effect for a share of patients, or a weak effect for everyone at this size. A
missing confounder weaker than 0.079 on both sides could not lower the rule's value by 5 per 100.

## Go deeper

### How we know the method works

Simulation studies check each method over many repeated samples.

| part of the page | simulation study | its data |
| --- | --- | --- |
| the learned rule | [learned-rule CV-TMLE](../technical-reference/method-evidence/learned-rule-cvtmle.md) (cross-validated TMLE) | 2,000 rows, ten folds and this page's outcome model. On simulated data of the same shape (a need-like score, a yes-or-no characteristic, and an effect that changes steadily with the score) the 95% intervals contained the true value in 0.9465 of the samples (the target is 0.95); on data where the outcome model is wrong, 0.9413 |
| the learned rule, at its limit | [learned-rule CV-TMLE at the boundary](../technical-reference/method-evidence/learned-rule-cvtmle-boundary.md) | simulated data where the offer has no effect for anyone, or a weak one. The interval covered only 0.8938 and 0.8948 |
| the random offer | [stochastic point-treatment regimes](../technical-reference/method-evidence/stochastic-point-treatment-regimes.md) | other simulated data, with one three-level characteristic, compared with `lmtp`, another package that fits the same estimate in R |

The random-offer study gives the method the true chances of the real offer, while this page
estimates them, so the study does not test the treatment model. The [technical
entry](../technical-reference/point-treatment-tmle.md#validation-issues-special-to-this-method)
lists the method's evidence and limits.

A simulation we ran for this page redraws the data 400 times at the page's size and refits each
sample with the page's settings. It checks the page's readings and does not validate the method.
Its script and log are in the `cleverly` repository at
`reviews/notebook-review/probes/treatment-rules/`. The learned rule's interval covered the true
value of its fold rules in 0.9475 of the samples, and its standard error matched the spread of
the errors (ratio 0.9918). The random offer's interval covered 0.9375.

### Try to break the result

These checks change the data in a known way and refit each fit 20 times. The placebo test
(`placebo`) shuffles the recorded offers among patients, so the share offered stays the same but
the offer can have no effect: the random offer's gain should then be near
zero, and every rule is worth the plain share who stayed home, 0.6695. The noise test
(`random_common_cause`) adds a pure-noise column, so the estimate should barely move. The subset
test (`subset`) refits on random 70% samples, so the refits should spread about as much as the
standard error predicts.

None can detect a missing confounder. For the learned rule, the noise and subset tests keep the
rule each patient got and the ten folds, so they check the estimate of these rules' value.

In [14]:
print(f"share who stayed home: {frame['home_30'].mean():.4f}")
columns = [
    "test",
    "original",
    "refuted_mean",
    "refuted_sd",
    "statistic_name",
    "statistic",
    "verdict",
]
refutations = {}
for label, fit, name in (
    ("random offer's gain", random_fit, RANDOM_OFFER),
    ("learned rule's value", learned, "ey_learned_rule[learned rule]"),
):
    refutations[label] = fit.diagnostics.refute(
        estimand=name, tests=("placebo", "subset", "random_common_cause"), random_state=61
    )
    print(label)
    print(refutations[label].to_frame()[columns].round(4).to_string(index=False))

share who stayed home: 0.6695


random offer's gain
               test  original  refuted_mean  refuted_sd                  statistic_name  statistic     verdict
            placebo    0.0262        0.0013      0.0067                               t     0.8566 descriptive
             subset    0.0262        0.0263      0.0048               scatter ratio rho     0.9421 descriptive
random_common_cause    0.0262        0.0264      0.0003 paired shift in standard errors     0.0230 descriptive


learned rule's value
               test  original  refuted_mean  refuted_sd                  statistic_name  statistic     verdict
            placebo    0.7174        0.6692      0.0139                               t    -0.1006 descriptive
             subset    0.7174        0.7161      0.0081               scatter ratio rho     0.9116 descriptive
random_common_cause    0.7174        0.7175      0.0005 paired shift in standard errors     0.0033 descriptive


With shuffled offers, the random offer's gain averages near zero and the rules are worth about the
plain share who stayed home; the `t` column, each placebo mean's distance from that expected value
in standard errors of the 20 refits' mean, is below 1 for both. The noise column barely moves either
estimate: the paired shift, its move in standard errors, is near 0. The subset refits spread about
as the standard error predicts: the scatter ratio, the spread divided by the predicted spread, is
near 1 for both, and 20 refits leave wide room around 1.

The verdict reads `descriptive`: `cleverly` gives a pass or fail only when a simulation study backs
the check for this kind of fit, and here none does, so you judge the numbers.

The [point-treatment example](point-treatment-tmle.ipynb) also tests the method on outcomes with a
known effect. For a learned rule, those tests have no simulated truth yet.

## Where to go next

The [interventions example](interventions.ipynb) compares a fixed screening rule and other plans
with offering to everyone. The
[point-treatment TMLE reference](../technical-reference/point-treatment-tmle.md#learned-rules)
states the conditions of the learned-rule value, and the
[estimands guide](../user-guide/estimands.md#learned-treatment-rules) shows what a learned-rule
result holds. The [examples index](index.md) lists every example.